# FASE 3 - Plano 2: EDA Geografica

**Pessoa 2** — Evidencia geografica do Ato 1

Objetivo: Produzir mapa coropletico por UF (bad_review_rate normalizado por taxa, nao volume) e heatmap de rotas origem x destino.

Outputs:
- `data/processed/geo_aggregated.parquet` — agregacao por UF para Streamlit (Phase 6)
- `reports/figures/eda03_choropleth_bad_reviews_uf.png` — mapa estatico para slides
- `reports/figures/eda03_choropleth_static.html` — mapa interativo folium
- `reports/figures/eda05_rotas_heatmap.png` — heatmap corredores origem x destino


## Celula 1 — Setup

In [ ]:
import pandas as pd
import geopandas as gpd
import folium
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from pathlib import Path

FIGURES = Path("reports/figures")
FIGURES.mkdir(parents=True, exist_ok=True)
PROCESSED = Path("data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)
DOCS = Path("docs")
DOCS.mkdir(parents=True, exist_ok=True)
GOLD = Path("data/gold/olist_gold.parquet")

print("Setup OK")

## Celula 2 — Load gold e derivacao defensiva dias_atraso

In [ ]:
df = pd.read_parquet(GOLD)

# Derivar dias_atraso defensivamente (formula padrao do contrato)
df["dias_atraso"] = (
    pd.to_datetime(df["order_delivered_customer_date"])
    - pd.to_datetime(df["order_estimated_delivery_date"])
).dt.days

print(f"Shape: {df.shape}")
print(f"customer_state valores unicos: {sorted(df['customer_state'].dropna().unique())}")
print(f"dias_atraso: mean={df['dias_atraso'].mean():.1f}, notna={df['dias_atraso'].notna().sum()}")

## Celula 3 — Agregacao geografica com schema completo

In [ ]:
geo_agg = (
    df.groupby("customer_state")
    .agg(
        total_orders=("order_id", "count"),
        bad_reviews=("bad_review", "sum"),
        avg_dias_atraso=("dias_atraso", "mean"),
        avg_freight_value=("freight_value", "mean"),
    )
    .assign(bad_review_rate=lambda x: x["bad_reviews"] / x["total_orders"])
    .reset_index()
)
print(geo_agg.sort_values("bad_review_rate", ascending=False).head(10))

# Export para Streamlit (Phase 6)
geo_agg.to_parquet(PROCESSED / "geo_aggregated.parquet", index=False)
print(f"Exportado: data/processed/geo_aggregated.parquet ({len(geo_agg)} UFs)")

## Celula 4 — Download e load do GeoJSON do Brasil

In [ ]:
GEOJSON_PATH = DOCS / "brazil_states.geojson"
GEOJSON_URL = "https://raw.githubusercontent.com/codeforamerica/click_that_hood/master/public/data/brazil-states.geojson"

if not GEOJSON_PATH.exists():
    r = requests.get(GEOJSON_URL, timeout=30)
    r.raise_for_status()
    GEOJSON_PATH.write_bytes(r.content)
    print(f"GeoJSON baixado: {GEOJSON_PATH}")
else:
    print(f"GeoJSON ja existe: {GEOJSON_PATH}")

br_states = gpd.read_file(str(GEOJSON_PATH))

# CRITICO: verificar campo de sigla do estado
print("Colunas do GeoJSON:", br_states.columns.tolist())
print("Primeira feature:", br_states.iloc[0].drop('geometry').to_dict())

## Celula 5 — Identificar campo de sigla e fazer merge

In [ ]:
# Detectar automaticamente o campo de sigla
possible_fields = ["sigla", "abbrev_state", "UF", "CD_UF", "SIGLA_UF"]
SIGLA_COL = None
for field in possible_fields:
    if field in br_states.columns:
        SIGLA_COL = field
        print(f"Campo de sigla encontrado: {SIGLA_COL}")
        break

if SIGLA_COL is None:
    print("ATENCAO: nenhum campo de sigla padrao encontrado. Colunas:", br_states.columns.tolist())
    raise ValueError("Ajuste SIGLA_COL manualmente para o campo correto")

# Nota: remover colunas com Timestamp antes do merge para compatibilidade com folium
br_states_clean = br_states.drop(columns=[
    c for c in br_states.columns
    if c != 'geometry' and br_states[c].dtype == 'datetime64[ns]'
], errors='ignore')

br_merged = br_states_clean.merge(geo_agg, left_on=SIGLA_COL, right_on="customer_state", how="left")
print(f"Merge: {br_merged['bad_review_rate'].notna().sum()} UFs com dados")

## Celula 6 — Choropleth interativo com folium

In [ ]:
m = folium.Map(location=[-15, -55], zoom_start=4, tiles="CartoDB positron")
folium.Choropleth(
    geo_data=br_merged.__geo_interface__,
    data=geo_agg,
    columns=["customer_state", "bad_review_rate"],
    key_on=f"feature.properties.{SIGLA_COL}",
    fill_color="YlOrRd",
    fill_opacity=0.75,
    line_opacity=0.3,
    legend_name="Taxa de Avaliacoes 1-2 Estrelas",
    nan_fill_color="lightgray",
).add_to(m)
folium.LayerControl().add_to(m)
m.save(str(FIGURES / "eda03_choropleth_static.html"))
print("Exportado: eda03_choropleth_static.html")

## Celula 7 — Choropleth estatico para slides

> **Nota:** Usa fallback geopandas/matplotlib (kaleido requer Chrome headless, nao disponivel neste ambiente).


In [ ]:
# Fallback: mapa estatico via geopandas plot
fig, ax = plt.subplots(figsize=(10, 12))
br_merged.plot(
    column="bad_review_rate",
    cmap="YlOrRd",
    legend=True,
    missing_kwds={"color": "lightgray"},
    ax=ax,
    edgecolor="white",
    linewidth=0.3,
    legend_kwds={"label": "Taxa de Avaliacoes 1-2 Estrelas"},
)
ax.set_title("Taxa de Avaliacoes 1-2 Estrelas por UF", fontsize=15, fontweight="bold")
ax.axis("off")
fig.savefig(FIGURES / "eda03_choropleth_bad_reviews_uf.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("PNG exportado via fallback geopandas")
fsize = (FIGURES / "eda03_choropleth_bad_reviews_uf.png").stat().st_size
print(f"PNG size: {fsize:,} bytes")

---

## EDA-05: Heatmap de Rotas Origem x Destino (top-15 origens)

## Celula 8 — Verificar disponibilidade de seller_state

In [ ]:
assert "seller_state" in df.columns, "seller_state ausente na gold table"
assert "dias_atraso" in df.columns, "dias_atraso ausente"

route_data = df.dropna(subset=["dias_atraso", "seller_state", "customer_state"]).copy()
print(f"Linhas com dados de rota: {len(route_data):,} de {len(df):,}")
print(f"seller_state valores unicos: {route_data['seller_state'].nunique()}")

## Celula 9 — Filtrar top-15 origens por volume (evita 27x27 ilegivel)

In [ ]:
top_origins = (
    route_data.groupby("seller_state")["order_id"]
    .count()
    .nlargest(15)
    .index
)
print(f"Top-15 origens (seller_state): {sorted(top_origins.tolist())}")

route_filtered = route_data[route_data["seller_state"].isin(top_origins)]

## Celula 10 — Pivot table de atraso medio por corredor

In [ ]:
pivot = route_filtered.pivot_table(
    values="dias_atraso",
    index="seller_state",
    columns="customer_state",
    aggfunc="mean",
)

print(f"Pivot shape: {pivot.shape} (origens x destinos)")
print(f"Atraso medio geral: {pivot.values[~__import__('numpy').isnan(pivot.values)].mean():.1f} dias")

## Celula 11 — Heatmap seaborn (slide export)

In [ ]:
sns.set_theme(style="white", context="talk")

fig, ax = plt.subplots(figsize=(18, 8))
sns.heatmap(
    pivot,
    cmap="RdYlGn_r",
    center=0,
    linewidths=0.3,
    linecolor="white",
    annot=False,
    cbar_kws={"label": "Atraso Medio (dias)", "shrink": 0.8},
    ax=ax,
)
ax.set_title(
    f"Atraso Medio por Corredor Vendedor (Origem) x Cliente (Destino)\n"
    f"Top-{len(top_origins)} origens por volume de pedidos",
    fontsize=14, fontweight="bold", pad=12,
)
ax.set_xlabel("UF Destino (cliente)", fontsize=12)
ax.set_ylabel("UF Origem (vendedor)", fontsize=12)
plt.xticks(rotation=45, ha="right", fontsize=10)
plt.yticks(rotation=0, fontsize=10)
fig.tight_layout()
fig.savefig(FIGURES / "eda05_rotas_heatmap.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Exportado: eda05_rotas_heatmap.png")
fsize2 = (FIGURES / "eda05_rotas_heatmap.png").stat().st_size
print(f"PNG size: {fsize2:,} bytes")

## Celula 12 — Top 10 corredores mais problematicos (insight de narrativa)

In [ ]:
corridor_avg = (
    route_filtered.groupby(["seller_state", "customer_state"])["dias_atraso"]
    .agg(["mean", "count"])
    .reset_index()
    .rename(columns={"mean": "atraso_medio", "count": "volume"})
    .query("volume >= 50")
    .sort_values("atraso_medio", ascending=False)
    .head(10)
)
print("Top 10 corredores mais atrasados (minimo 50 pedidos):")
print(corridor_avg.to_string(index=False))